# Structured Output & Tool Use- Make the Model Talk to Code

**Module 5 · Lesson 5.5**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- From free text to structured data - the reliability ladder
- Schema-constrained output - guaranteed JSON
- Validate at the boundary - syntax vs semantics
- Tool use - let the model call your functions
- Many tools, parallel calls, and routing
- Cross-provider and production - instructor, MCP, the project

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The whole game: from "trust me, it's JSON" to a guarantee

> 📝 **Analogy**
>
> **Structured output is handing the model a form, not having a conversation.** A form has typed fields - name, rating, in stock - and you get back exactly those fields, filled in, every time. No prose, no markdown fence, nothing to clean up before your code can use it.
> **Tool use is giving an assistant a phone and a rolodex.** The assistant does not do the task itself - it tells you which number to call and what to say; you make the call and read back the result. A model that can call your functions stops being a chatbot and starts being a system that does things.
> **Where the analogy breaks down:** a filled form is guaranteed to have the right *boxes*, not the right *answers* - schema-valid is not fact-valid. And the assistant can ask you to dial a number that does not exist, so your code must handle a tool call that errors or returns nothing.

Every technique here is tested with **real API calls** and paired with a plain-English walkthrough. We use Gemini through the current unified SDK, but structured output and tool use are now standard across Claude and GPT.

- **Generate** schema-constrained JSON with a Pydantic schema and native structured output, and explain why constrained decoding guarantees valid syntax.

- **Apply** Pydantic validation at the boundary - distinguishing guaranteed syntax from unguaranteed semantics - and retry on failure.

- **Construct** a function-calling loop where the model selects a tool, your code executes it, and the result returns for a final answer.

- **Evaluate** when to use prompt-trick JSON, JSON mode, schema-constrained output, or tool calling, and choose across providers.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier lessons. Each one is load-bearing for today.

## The setup: one helper we will reuse all lesson

Examples call this `client` on the current unified **google-genai** SDK (the older `google.generativeai` package was deprecated in 2025 - [migration guide](https://ai.google.dev/gemini-api/docs/migrate)). We will pass schemas and tools through its `config`.

**📝 `setup.py Gemini`** - *API*

In [ ]:
# pip install "google-genai>=1.0.0" pydantic
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import os, json

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-3-flash"

def generate(prompt: str, config: types.GenerateContentConfig = None):
    """Thin wrapper so every example shares one error-handled call."""
    try:
        return client.models.generate_content(model=MODEL, contents=prompt, config=config)
    except Exception as e:
        raise RuntimeError(f"API call failed: {e}")

print(generate("Reply with one word: ready").text)
# Output: ready

- `genai.Client(...)` - one reusable client (the deprecated package used a global `configure()` plus per-call `GenerativeModel`).

- **Everything rides on `config`.** Structured output and tools are both passed through `types.GenerateContentConfig` - a schema in Step 2, a list of tool functions in Step 4.

- We import `pydantic.BaseModel` now because it is the lingua franca for both schemas and tool signatures in 2026.

- The wrapper turns a network blip into one clear error instead of crashing a multi-step pipeline.

---

## 🎯 Concept 1: From free text to structured data - the reliability ladder

### From free text to structured data - the reliability ladder

Why machine-readable output matters, and the three rungs from hope to guarantee.

**"Trust me" versus a signed contract.** A friend promising to show up is not the same as a calendar invite they accepted. "The model said JSON" is the promise; a schema is the contract.

The gain: you stop hoping and start guaranteeing. The ladder goes from a prompt request, to valid-JSON mode, to schema-guaranteed output.

> 📦 **Info**
>
> The reliability ladder
> - **Prompt-trick JSON** - "Return ONLY JSON." No guarantee; prose, fences, and bad commas slip through. The 2023 stopgap.
> - **JSON mode** - the API guarantees the output is *valid JSON*, but not that it has your fields and types.
> - **Schema-constrained output** - constrained decoding (the model generates one token at a time, and at each step the API blocks any token that would break your schema) makes the output match your JSON Schema every time. Valid syntax *and* the right shape. The 2026 production default.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened? Free text is not machine-usable, and "Return ONLY JSON" is a hope, not a guarantee. The ladder runs from prompt-trick JSON (fragile) through JSON mode (valid but unshaped) to schema-constrained output (guaranteed shape). **In 2026 production lives at the top rung** - the rest of this step is why, and the next is how.

---

## 🎯 Concept 2: Schema-constrained output - guaranteed JSON

### Schema-constrained output - guaranteed JSON

Hand the model a Pydantic schema and get back a validated object, every time.

**📝 `structured_output.py Gemini`** - *API*

In [ ]:
class Product(BaseModel):
    name: str
    rating: int = Field(ge=1, le=5)
    in_stock: bool
    tags: list[str]

resp = generate(
    "Extract a product record from: 'The Buds 3 are great, 4 stars, in stock. #audio #anc'",
    types.GenerateContentConfig(
        response_mime_type="application/json",   # JSON mode...
        response_schema=Product,                  # ...constrained to THIS schema
    ),
)
product = resp.parsed          # already a validated Product instance
print(product.name, product.rating, product.tags)
# Output: Buds 3 4 ['audio', 'anc']

- **The Pydantic model IS the schema.** `Product` declares the field names, types, and constraints (`rating` must be an int in 1-5); the SDK converts it to a JSON Schema for you.

- **`response_schema=Product`** turns on constrained decoding: at every token the decoder can only pick tokens that keep the output matching the schema, so an int field can never receive a string.

- **`resp.parsed`** hands you a ready-made `Product` instance - no `json.loads`, no try/except around parsing, because the output is guaranteed to parse and match.

- This is the same "show the exact shape" instinct from Lesson 5.1, promoted from a prompt request to an enforced contract.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 💡 **Info**
>
> ⚠️ Common mistake Treating a schema-valid object as a correct object. Constrained decoding guarantees the JSON parses and matches your types - it does **not** guarantee the values are right. A `rating` will always be 1-5, but it can be the wrong number. Guaranteed shape, not guaranteed truth.

#### 💡 What just happened

⚡ What just happened? Passing a Pydantic model as `response_schema` turns on constrained decoding: the output is guaranteed to parse and match your fields and types, and `resp.parsed` hands you a typed object directly. **This is the top rung - valid syntax, every single time** - and it is now native across Gemini, OpenAI (`json_schema` strict), and Claude.

---

## 🎯 Concept 3: Validate at the boundary - syntax vs semantics

### Validate at the boundary - syntax vs semantics

Constrained decoding gives you the shape. You still check the meaning.

**📝 `validate_boundary.py`** - *Production*

In [ ]:
from pydantic import field_validator, ValidationError

class Invoice(BaseModel):
    gstin: str
    amount: float = Field(gt=0)

    @field_validator("gstin")
    @classmethod
    def check_gstin(cls, v: str) -> str:
        if not (len(v) == 15 and v[:2].isdigit()):   # real check is stricter
            raise ValueError("not a valid GSTIN")
        return v

def extract_invoice(text: str, retries: int = 2) -> Invoice:
    for attempt in range(retries + 1):
        resp = generate(f"Extract invoice fields from: {text}",
            types.GenerateContentConfig(response_mime_type="application/json", response_schema=Invoice))
        try:
            # parse the raw JSON ourselves so the validators run and RAISE
            return Invoice.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == retries: raise
            # else: loop; a stronger version appends str(e) to the prompt

inv = extract_invoice("GSTIN 29ABCDE1234F1Z5, total 4500.00")
print(inv.gstin, inv.amount)
# Output: 29ABCDE1234F1Z5 4500.0

- **The schema handles syntax;** the `field_validator` handles *meaning* - it checks the GSTIN actually looks like a GSTIN, which no type constraint can express.

- **Parse the raw JSON yourself so validators raise.** A subtle trap: `resp.parsed` *swallows* a validator failure and hands back `None`, so the retry never fires. `Invoice.model_validate_json(resp.text)` runs the validators and *raises* `ValidationError` - which is what makes the retry loop real.

- **The retry loop** gives the model another chance on a semantic miss: because sampling is non-deterministic, re-running the identical prompt can return a different (valid) value - and a stronger version appends the error text so the model self-corrects. It gives up cleanly after `retries`, the same bounded-loop discipline as the reliability harness in Lesson 5.1.

- The legacy India formats (GSTIN, PAN, Aadhaar) are the classic case: the model emits a plausible-looking string that fails a checksum, so the boundary validator is non-negotiable.

#### 💡 What just happened

⚡ What just happened? Schema-constrained output guarantees the shape; a Pydantic validator at the boundary guarantees the *meaning*, raising on a value that looks right but is not, with a bounded retry. **Guaranteed syntax plus validated semantics is the production contract** - never trust raw values just because the JSON parsed.

---

## 🎯 Concept 4: Tool use - let the model call your functions

### Tool use - let the model call your functions

The model proposes a function call; your code runs it and hands back the result.

> 📞 **Analogy**
>
> **A dispatcher and a field crew.** The dispatcher (the model) decides which job to send and with what details, but does not do the fieldwork. The crew (your code) executes and radios back the result, and the dispatcher writes up the outcome.
> **Where it breaks down:** a dispatcher can send a crew to a wrong or nonexistent address - the model can request a tool that fails or returns nothing - so your code must handle a bad call, not blindly trust it.

**📝 `tool_use.py Gemini`** - *API*

In [ ]:
def get_stock(sku: str) -> int:
    """Return the number of units in stock for a product SKU."""
    return {"buds-3": 42, "case-x": 0}.get(sku, 0)

# Pass the Python function as a tool. The SDK reads the signature + docstring
# to build the schema, and Automatic Function Calling runs it for you.
resp = generate(
    "How many buds-3 do we have in stock?",
    types.GenerateContentConfig(tools=[get_stock]),
)
print(resp.text)
# Output: You have 42 units of buds-3 in stock.

- **Your Python function is the tool.** Its type hints and docstring become the tool's schema - the SDK generates the JSON Schema from `get_stock(sku: str) -> int` automatically.

- **The model emits a call, not an answer.** Internally it returns `get_stock(sku="buds-3")` - structured arguments, typed to your signature.

- **Automatic Function Calling closes the loop.** Passing `tools=[get_stock]` lets the SDK execute the function, feed the result (42) back to the model, and continue to the final answer - all in one `generate` call.

- Want manual control? You can disable AFC and run the loop yourself - get the tool call, execute it, append the result, call again - which is exactly the ReAct loop from Lesson 5.3.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened? A tool is just your function with a docstring: the SDK turns it into a schema, the model decides to call it and emits typed arguments, and Automatic Function Calling runs it and feeds the result back for the final answer. **The model proposes the call; your code disposes** - this is the bridge from chatbot to a system that acts, and the foundation of agents (Module 11).

---

## 🎯 Concept 5: Many tools, parallel calls, and routing

### Many tools, parallel calls, and routing

Give the model several tools; it picks the right ones - sometimes more than one at once.

**📝 `parallel_tools.py Gemini`** - *API*

In [ ]:
def get_stock(sku: str) -> int:
    """Units in stock for a SKU."""
    return {"buds-3": 42}.get(sku, 0)

def get_price(sku: str) -> int:
    """Price in rupees for a SKU."""
    return {"buds-3": 3299}.get(sku, 0)

resp = generate(
    "How many buds-3 are in stock and what do they cost?",
    types.GenerateContentConfig(tools=[get_stock, get_price]),   # model picks and routes
)
print(resp.text)
# Output: There are 42 buds-3 in stock at 3299 rupees each.
# (the model called BOTH tools in parallel, in one turn)

- **Give it a toolbox, not one tool.** Pass a list; the model reads each docstring and *routes* the request to the right tool(s) - it would call only `get_price` for a price-only question.

- **Parallel calls collapse round-trips.** For "stock and price" it emits both calls at once; the SDK runs them (you can execute them concurrently) and returns both results together, so it is one extra turn, not two.

- **Forced choice when you need it.** A `tool_config` can require the model to call a specific tool (or any tool) rather than answer directly - useful when an action is mandatory.

- More tools means more routing decisions, so keep tool names and docstrings crisp and non-overlapping or the model will pick wrong.

#### 💡 What just happened

⚡ What just happened? With several tools, the model becomes a router: it reads the docstrings and calls the right one - or several in parallel - and you can force a specific tool when an action is mandatory. **Parallel calls turn a multi-step task into one round-trip**, which is the difference between a snappy and a sluggish tool-using app.

---

## 🎯 Concept 6: Cross-provider and production - instructor, MCP, the project

### Cross-provider and production - instructor, MCP, the project

The same patterns across Gemini, OpenAI, and Claude - and where they grow up.

> 📦 **Info**
>
> Same idea, three surfaces
> - **Gemini:** `response_schema` + a Pydantic model; tools as Python functions with Automatic Function Calling.
> - **OpenAI:** `response_format: {type: "json_schema", strict: true}`; tools via the function-calling API, parallel by default.
> - **Claude:** constrained-decoding structured output (GA early 2026) and a tool-use API with the same propose-execute-return loop.
> The **instructor** library gives you one Pydantic-first call across all three; **MCP** (Model Context Protocol) is the emerging standard for exposing tools to any model - both deepened in **Module 10**.

**📝 `product_api.py Complete`** - *project*

In [ ]:
# Project: a product endpoint that extracts a validated record AND can
# answer live stock questions via a tool - structured output + tool use together.
class Product(BaseModel):
    name: str
    rating: int = Field(ge=1, le=5)
    in_stock: bool

def stock_lookup(sku: str) -> int:
    """Live units in stock for a SKU."""
    return {"buds-3": 42}.get(sku, 0)

def extract_product(text: str) -> Product:
    return generate(f"Extract a product record from: {text}",
        types.GenerateContentConfig(response_mime_type="application/json",
                                    response_schema=Product)).parsed

def answer_query(question: str) -> str:
    return generate(question, types.GenerateContentConfig(tools=[stock_lookup])).text

print(extract_product("Buds 3, 4 stars, in stock"))
# Output: name='Buds 3' rating=4 in_stock=True
print(answer_query("Is buds-3 in stock right now?"))
# Output: Yes - there are 42 units of buds-3 in stock.

- **`extract_product`** is pure structured output: free text in, a validated `Product` out via `response_schema` - no parsing code.

- **`answer_query`** is pure tool use: the model routes a live question to `stock_lookup` and answers from the real result.

- **Together they are the pattern** behind real product APIs: structured extraction to populate your database, tool calling to answer live questions against it.

- Swap Gemini for OpenAI or Claude and only these two config surfaces change; the `Product` schema and the tool function are reused as-is.

Shipping `data = json.loads(resp.text)` on a "Return ONLY JSON" prompt, with no schema and no validation. It passes every test, then crashes in production the first time the model adds a friendly "Here you go:" or wraps the JSON in a markdown fence. The fix is the whole lesson: schema-constrain the output, validate the values, and let the model call tools for anything it cannot know - never parse-and-pray.

#### 💡 What just happened

⚡ What just happened? The same two patterns - schema-constrained output and tool calling - are native across Gemini, OpenAI, and Claude, wrapped to near one-line by `instructor` and standardized for tools by MCP. The project combined both: structured extraction to store data, tool use to answer live questions. **This is how a model stops talking and starts plugging into your systems.**

## Putting it together: the talk-to-code toolkit

| Technique | Use it when | Guarantee |
|---|---|---|
| Prompt-trick JSON | A throwaway script, no downstream parser | None |
| JSON mode | You need valid JSON, shape is flexible | Valid JSON |
| Schema-constrained output | Code consumes the output (the 2026 default) | Shape (syntax) |
| Pydantic validation | Values must be semantically correct | Meaning |
| Tool calling | The model must act or fetch live data | Action via your code |
| Parallel tools | One request needs several tools | One round-trip |

### 🧮 The whole lesson on one screen

**Climb the ladder** from prompt-trick JSON to schema-constrained output (Step 1); **guarantee the shape** with a Pydantic `response_schema` (Step 2); **validate the meaning** at the boundary, because shape is not truth (Step 3); **let the model call your functions** - it proposes, your code disposes (Step 4); **give it a toolbox** and let it route and call in parallel (Step 5); and **carry it across providers** with the same patterns (Step 6). Above all: schema-constrain, validate, and never parse-and-pray.

Forward hooks you just planted: we'll build the full tool ecosystem and MCP in Module 10; agents that chain many tool calls grow up in Module 11; retrieval becomes a tool the model calls in Module 8 (RAG); and we'll come back to the cost of constrained decoding and tool loops in Module 13.

- Schick et al., *Toolformer: Language Models Can Teach Themselves to Use Tools* (2023) - [arxiv.org/abs/2302.04761](https://arxiv.org/abs/2302.04761); Patil et al., *Gorilla* (2023) - [arxiv.org/abs/2305.15334](https://arxiv.org/abs/2305.15334)

- Yao et al., *ReAct: Synergizing Reasoning and Acting* (2022) - [arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629)

- Geng et al., *Grammar-Constrained Decoding for Structured NLP* (2023) - [arxiv.org/abs/2305.13971](https://arxiv.org/abs/2305.13971)

- Geng et al., *JSONSchemaBench* (2025) - [arxiv.org/abs/2501.10868](https://arxiv.org/abs/2501.10868)

- OpenAI [Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs); Google [structured output](https://ai.google.dev/gemini-api/docs/structured-output) & [function calling](https://ai.google.dev/gemini-api/docs/function-calling); Anthropic [tool use](https://platform.claude.com/docs/en/build-with-claude/tool-use)

- [Pydantic](https://docs.pydantic.dev) and the [instructor](https://python.useinstructor.com) library; Google [google-genai migration guide](https://ai.google.dev/gemini-api/docs/migrate)

## Key takeaways

> ℹ️ **Info**
>
> What you learned - 6 ideas
> - **The reliability ladder** - prompt-trick JSON to JSON mode to schema-constrained; production lives at the top.
> - **Schema-constrained output** - a Pydantic `response_schema` guarantees valid syntax every time, and `resp.parsed` hands you a typed object.
> - **Validate at the boundary** - constrained decoding gives shape, not truth; a field validator catches semantically wrong values.
> - **Tool use** - the model emits a typed tool call; your code executes it and feeds the result back. Propose, then dispose.
> - **Many tools, parallel calls** - the model routes across a toolbox and calls several at once, collapsing round-trips.
> - **Cross-provider** - the same patterns on Gemini, OpenAI, and Claude; instructor and MCP standardize them.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Structured Output & Tool Use- Make the Model Talk to Code**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-5_5.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-5.5.html` - regenerate this notebook whenever the source HTML is updated.*
